# Sensor Fusion: Calibration, Association and Late Fusion

**Module 01 · Perception and Prediction** · Notebook 6 of 9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/06_sensor_fusion.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate | [03 LiDAR and point clouds](03_lidar_point_clouds.ipynb), [04 Object detection](04_object_detection.ipynb) | ISO 26262-9:2018; ISO 21448:2022; ISO/PAS 8800:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Explain what sensor fusion can and cannot contribute to safety, and distinguish early, late and BEV feature-level fusion
2. Project LiDAR points and 3D boxes into a camera image with real KITTI calibration
3. Build a camera–LiDAR late fusion on real data: a pretrained 2D detector, LiDAR clusters, frustum association and depth from LiDAR
4. Evaluate single-sensor and fused object lists against KITTI 3D labels and draw conclusions from the measured numbers
5. Measure how extrinsic calibration errors destroy association, and estimate time-synchronisation errors for the KITTI rig
6. Turn the measured weaknesses into inputs for a fusion design FMEA

## Setup

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from scipy.optimize import linear_sum_assignment
from scipy.spatial import ConvexHull
from scipy.spatial.transform import Rotation
from sklearn.cluster import DBSCAN
from IPython.display import display
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn

torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(0)
rng = np.random.default_rng(42)
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
METHOD_COLOURS = {'Camera': OKABE_ITO[0], 'LiDAR': OKABE_ITO[2], 'Fusion AND': OKABE_ITO[4],
                  'Fusion camera-led': OKABE_ITO[6], 'Fusion OR': OKABE_ITO[5]}
CLASS_COLOURS = {'Vehicle': OKABE_ITO[4], 'Pedestrian': OKABE_ITO[5], 'Cyclist': OKABE_ITO[0]}
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(f"NumPy {np.__version__}, torch {torch.__version__}, torchvision {torchvision.__version__}, avsafety {avsafety.__version__}")

### Data and licence

Camera images, LiDAR scans, calibration and 3D labels come from the **KITTI object-detection benchmark** (Geiger et al., 2012, 2013); `FRAMES` is the same seeded draw of 30 frames used in notebooks 02 and 03 (each frame has at least two cars and at least two pedestrians or cyclists). The camera detector is torchvision's Faster R-CNN with a MobileNetV3-Large FPN backbone, pretrained on COCO (weights downloaded once from download.pytorch.org, about 75 MB). It has **never seen KITTI**: we use it as an off-the-shelf component, which is realistic for a late-fusion architecture built from supplier modules — and it means the absolute numbers below are lower than a KITTI-trained detector would achieve.

In [ ]:
print(D.attribution('KITTI'))
FRAMES = [35, 134, 217, 274, 462, 1134, 1868, 2096, 2227, 3078, 3677, 3687, 4170, 4283, 4292,
          4520, 4651, 4755, 4804, 4972, 5259, 5430, 5622, 5838, 5920, 5934, 6333, 6511, 6785, 6916]
SHOW = 4804                        # showcase frame: pedestrians and cyclists crossing at 13-30 m
frames = {}
for f in FRAMES:
    try:
        frames[f] = D.kitti_object(f)
    except D.DataUnavailable as exc:
        print(f"frame {f} skipped: {exc}")
if not frames:
    raise SystemExit("KITTI could not be downloaded. Run the notebook with internet access.")
print(f"Loaded {len(frames)} frames: {sum(len(fr.points) for fr in frames.values()):,} LiDAR points, "
      f"{sum(l['type'] != 'DontCare' for fr in frames.values() for l in fr.labels)} labelled objects")

---
## 1. Why sensor fusion?

No single sensor is sufficient in all conditions (see [02 Sensor modalities](02_sensor_modalities.ipynb)):
camera — semantics but weak in darkness and glare; LiDAR — accurate 3D but degraded by fog and spray; radar — weather-robust velocity but low resolution.

**Potential benefits**
1. **Redundancy:** if one sensor fails or is degraded, others can compensate.
2. **Complementary strengths:** the camera classifies, LiDAR measures geometry, radar measures velocity.
3. **Fewer false positives and negatives** — *if* association and decision logic are designed well.
4. **Weather robustness:** diverse physics reduces (but does not eliminate) common-cause degradation.
5. **Safety argument:** redundant, diverse sensors can support ASIL decomposition (ISO 26262-9, which requires sufficient independence) and the mitigation of performance insufficiencies (ISO 21448). Neither standard mandates fusion or a particular sensor set.

Whether these benefits materialise is an empirical question. Section 5 measures it on 30 real KITTI frames instead of assuming it.

---
## 2. Fusion architectures

### Early fusion (data level)
Raw data are combined before detection, e.g. LiDAR points painted with image colours or depth channels added to images.
**Pros:** preserves information. **Cons:** very sensitive to calibration and synchronisation; heavy computation.

### Late fusion (decision / object level)
Each sensor runs its own detector; the **object lists** are associated and combined.
```
Camera -> detections \
                      -> association -> fused object list
LiDAR  -> detections /
```
**Pros:** modular, supplier-friendly, each sensor path can be validated independently. **Cons:** information already discarded by each detector; conflicts must be resolved by rules.
Historically common in production ADAS; BEV feature-level fusion is increasingly used. **We implement late fusion below, on real data.**

### Intermediate / feature-level ("deep") fusion
Learned features from each sensor are fused inside the network. Since about 2022 the dominant design transforms all sensors into a common **bird's-eye-view (BEV)** feature map:

- **BEVFormer** (Li et al., ECCV 2022): camera-only; spatio-temporal transformer that lifts multi-camera features into BEV.
- **BEVFusion** (Liu et al., ICRA 2023): camera and LiDAR features fused in a shared BEV representation for detection and map segmentation.
- **TransFusion** (Bai et al., CVPR 2022): transformer decoder that uses LiDAR queries and attends to image features (robust to calibration error).

**Pros:** best accuracy on current benchmarks (e.g. nuScenes). **Cons:** large paired datasets, end-to-end training, harder to validate each sensor path independently — an important consideration for the safety argument.

In [ ]:
fusion_comparison = pd.DataFrame({
    'Approach': ['Early fusion', 'Late fusion', 'Intermediate (BEV) fusion'],
    'Fusion level': ['Raw data', 'Object lists', 'Learned features'],
    'Calibration sensitivity': ['Very high', 'Medium', 'High (some designs tolerate errors)'],
    'Modularity / independent validation': ['Low', 'High', 'Low-medium'],
    'Typical use (as of 2026)': ['Research, some point painting', 'Production ADAS, modular stacks',
                                 'State of the art on benchmarks; increasingly in production'],
})
display(fusion_comparison)
for _, r in fusion_comparison.iterrows():
    print(f"  {r['Approach']}: fuses {r['Fusion level']}; calibration sensitivity {r['Calibration sensitivity']}; "
          f"modularity {r['Modularity / independent validation']}; use: {r['Typical use (as of 2026)']}")

---
## 3. Camera–LiDAR projection with real calibration

KITTI provides, per frame, the extrinsic transform from the LiDAR to the reference camera `Tr_velo_to_cam`, the rectification `R0` and the projection matrix `P2` of the left colour camera (intrinsics $f_x, f_y, c_x, c_y$ plus the stereo baseline offset):

$$ \tilde{\mathbf u} = P_2\, R_0\, T_{velo\to cam}\, \tilde{\mathbf p}_{velo}, \qquad u = \tilde u_1/\tilde u_3,\; v = \tilde u_2/\tilde u_3 $$

Intrinsics are typically estimated with Zhang's checkerboard method, extrinsics with targets or targetless methods. A first, cheap **calibration validation**: project the labelled 3D boxes into the image and compare them with the independently annotated 2D boxes. If calibration and labels are consistent, the two boxes should overlap closely.

In [ ]:
def project_rect(frame, rect):
    homo = np.c_[rect, np.ones(len(rect))]
    uvw = (frame.calib['P2'] @ homo.T).T
    return uvw[:, :2] / uvw[:, 2:3]

def iou2d(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

consistency = []
for f, fr in frames.items():
    H, W = fr.image.shape[:2]
    for l in fr.labels:
        if l['type'] == 'DontCare' or l['truncated'] > 0.1 or l['z'] < 2:
            continue
        uv = project_rect(fr, D.box3d_corners(l).T)
        box = [max(uv[:, 0].min(), 0), max(uv[:, 1].min(), 0), min(uv[:, 0].max(), W - 1), min(uv[:, 1].max(), H - 1)]
        consistency.append({'type': l['type'], 'distance_m': np.hypot(l['x'], l['z']),
                            'iou': iou2d(box, [l['x1'], l['y1'], l['x2'], l['y2']])})
consistency = pd.DataFrame(consistency)

fr = frames[SHOW]
uv_pts, depth, _ = fr.lidar_in_image()
fig, ax = plt.subplots(figsize=(15, 5.5))
ax.imshow(fr.image)
ax.scatter(uv_pts[:, 0], uv_pts[:, 1], c=depth, s=0.6, cmap='viridis_r', vmin=0, vmax=50, alpha=0.6)
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
for l in fr.labels:
    if l['type'] == 'DontCare' or l['z'] < 2:
        continue
    uv = project_rect(fr, D.box3d_corners(l).T)
    for a, b in edges:
        ax.plot(uv[[a, b], 0], uv[[a, b], 1], color=OKABE_ITO[5], lw=1)
    ax.add_patch(Rectangle((l['x1'], l['y1']), l['x2'] - l['x1'], l['y2'] - l['y1'], fill=False, ls='--',
                           edgecolor='white', lw=1))
ax.set_xlim(0, fr.image.shape[1]); ax.set_ylim(fr.image.shape[0], 0); ax.grid(False)
ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')
ax.set_title(f'Frame {fr.index}: LiDAR points (colour = depth), labelled 3D boxes projected with P2·R0·Tr (vermilion), '
             'annotated 2D boxes (white dashed)', fontsize=10)
plt.tight_layout(); plt.show()

print("--- Calibration / label consistency (text summary, measured) ---")
print(f"  {len(consistency)} non-truncated objects: IoU(projected 3D box, annotated 2D box) median "
      f"{consistency['iou'].median():.2f}, 5th percentile {consistency['iou'].quantile(0.05):.2f}")
print(f"  Objects with IoU < 0.7: {(consistency['iou'] < 0.7).sum()} "
      "(mostly small, distant objects where one pixel of disagreement matters)")
P2 = fr.calib['P2']
print(f"  Intrinsics of camera 2: fx = {P2[0, 0]:.1f} px, fy = {P2[1, 1]:.1f} px, cx = {P2[0, 2]:.1f} px, cy = {P2[1, 2]:.1f} px")

---
## 4. Two independent detection paths

### 4.1 Camera: a COCO-pretrained 2D detector

The detector outputs COCO classes. We map them to the three KITTI evaluation groups: car, bus and truck → **Vehicle**; person → **Pedestrian**; a person whose box overlaps a bicycle or motorcycle box (more than 30 % of the person box) → **Cyclist** (KITTI's cyclist box contains rider and bicycle); a bicycle without a rider → Cyclist as well, because the detector cannot tell parked from ridden bicycles — a known source of false alarms. The image is resized internally so that its short side is `MIN_SIZE` pixels; a smaller value is faster on a CPU but loses small, distant objects.

A monocular camera measures no depth. A classic estimate assumes a **flat road**: the bottom edge of the box touches the ground, so with camera height $h_c$ (1.65 m in KITTI) and the image row $v_b$ of the bottom edge,

$$ z \approx \frac{f_y\, h_c}{v_b - c_y} $$

The error grows quickly with distance: at 40 m one pixel of the bottom edge corresponds to more than 2 m of depth.

In [ ]:
MIN_SIZE = 600              # short side of the network input (px); KITTI images are 375 px high
SCORE_THR = 0.5
CAM_HEIGHT = 1.65                    # KITTI camera mounting height (m)
COCO_TO_GROUP = {3: 'Vehicle', 6: 'Vehicle', 8: 'Vehicle', 1: 'Pedestrian', 2: 'bike', 4: 'bike'}
GROUP = {'Car': 'Vehicle', 'Van': 'Vehicle', 'Truck': 'Vehicle', 'Pedestrian': 'Pedestrian',
         'Person_sitting': 'Pedestrian', 'Cyclist': 'Cyclist'}

detector = fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT', min_size=MIN_SIZE, max_size=4 * MIN_SIZE).eval()

def camera_detections(image):
    x = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
    with torch.no_grad():
        out = detector([x])[0]
    raw = [(b.numpy(), float(s), COCO_TO_GROUP[int(c)]) for b, s, c in zip(out['boxes'], out['scores'], out['labels'])
           if int(c) in COCO_TO_GROUP and s >= SCORE_THR]
    bikes = [r for r in raw if r[2] == 'bike']
    dets, used = [], set()
    for b, s, g in raw:
        if g == 'Vehicle':
            dets.append({'box': b, 'score': s, 'cls': 'Vehicle'})
        elif g == 'Pedestrian':
            rider = None
            for k, (bb, _, _) in enumerate(bikes):
                ix = max(0, min(b[2], bb[2]) - max(b[0], bb[0])); iy = max(0, min(b[3], bb[3]) - max(b[1], bb[1]))
                if k not in used and ix * iy > 0.3 * (b[2] - b[0]) * (b[3] - b[1]):
                    rider = k; break
            if rider is None:
                dets.append({'box': b, 'score': s, 'cls': 'Pedestrian'})
            else:
                used.add(rider); bb = bikes[rider][0]
                dets.append({'box': np.r_[np.minimum(b[:2], bb[:2]), np.maximum(b[2:], bb[2:])], 'score': s, 'cls': 'Cyclist'})
    dets += [{'box': bb, 'score': s, 'cls': 'Cyclist'} for k, (bb, s, _) in enumerate(bikes) if k not in used]
    return dets

def flat_ground_point(frame, box):
    """Camera-only BEV position (x, z) in the rectified camera frame from the flat-road assumption."""
    P2 = frame.calib['P2']
    fx, fy, cx, cy = P2[0, 0], P2[1, 1], P2[0, 2], P2[1, 2]
    if box[3] <= cy + 2:
        return None                                  # bottom edge at or above the horizon
    z = fy * CAM_HEIGHT / (box[3] - cy)
    return np.array([((box[0] + box[2]) / 2 - cx) * z / fx, z])

t0 = time.time()
cam = {f: camera_detections(fr.image) for f, fr in frames.items()}
t_cam = (time.time() - t0) / len(frames)
n_cls = pd.Series([d['cls'] for dl in cam.values() for d in dl]).value_counts()
print(f"Camera detector: {t_cam:.2f} s per frame on {torch.get_num_threads()} CPU threads; "
      f"{sum(map(len, cam.values()))} detections with score >= {SCORE_THR}: " + ", ".join(f"{k} {v}" for k, v in n_cls.items()))

### 4.2 LiDAR: class-agnostic clusters

The LiDAR path is the cluster pipeline from [03 LiDAR and point clouds](03_lidar_point_clouds.ipynb), condensed: RANSAC ground plane, DBSCAN on the points 0.25–3 m above the road, size gate (length ≤ 12 m, height ≥ 0.5 m). It knows *where* something is but not *what* it is. Only clusters with most of their points inside the camera image are kept, because KITTI labels exist only there.

In [ ]:
def ransac_ground(points, rng, n_iter=150, thr=0.15):
    cand = points[(points[:, 2] < -1.2) & (np.abs(points[:, 1]) < 20) & (points[:, 0] > 0) & (points[:, 0] < 50), :3]
    best, best_n = None, -1
    for _ in range(n_iter):
        s = cand[rng.choice(len(cand), 3, replace=False)]
        n = np.cross(s[1] - s[0], s[2] - s[0]); norm = np.linalg.norm(n)
        if norm < 1e-6 or abs(n[2]) / norm < 0.9:
            continue
        n = n / norm; d = -n @ s[0]
        k = (np.abs(cand @ n + d) < thr).sum()
        if k > best_n:
            best, best_n = (n, d), k
    n, d = best
    inl = cand[np.abs(cand @ n + d) < thr]
    a, b, c = np.linalg.lstsq(np.c_[inl[:, :2], np.ones(len(inl))], inl[:, 2], rcond=None)[0]
    n = np.array([-a, -b, 1.0]); norm = np.linalg.norm(n)
    return n / norm, -c / norm

def lidar_clusters(points, rng, eps=0.5, min_samples=8):
    """Clusters of non-ground points (LiDAR frame). Each: points (N x 4), extent, height."""
    n, d = ransac_ground(points, rng)
    hz = points[:, :3] @ n + d
    m = (hz > 0.25) & (hz < 3.0) & (points[:, 0] > 0) & (points[:, 0] < 55) & (np.abs(points[:, 1]) < 30)
    q, hq = points[m], hz[m]
    lab = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(q[:, :2])
    out = []
    for k in range(lab.max() + 1):
        sel = lab == k
        xy = q[sel, :2]
        try:
            hull = xy[ConvexHull(xy).vertices]
            length = max(np.linalg.norm(a - b) for a in hull for b in hull)
        except Exception:
            length = np.ptp(xy, axis=0).max()
        if length <= 12 and hq[sel].max() >= 0.5:
            out.append({'points': q[sel], 'n': int(sel.sum())})
    return out, (n, d)

t0 = time.time()
lidar = {f: lidar_clusters(fr.points, rng)[0] for f, fr in frames.items()}
print(f"LiDAR clustering: {(time.time() - t0) / len(frames):.2f} s per frame; "
      f"{sum(map(len, lidar.values()))} object-like clusters in total")

---
## 5. Late fusion on real data

### 5.1 Frustum association

Each camera box defines a **frustum**: the 3D wedge of space that projects into the box (Qi et al., 2018 use the same idea with a learned network). We project every cluster's points into the image with the calibration and count how many fall inside the inner part of each camera box (shrunk by 15 % on each side, so that background points at the box edges count less). The association score of camera box $i$ and cluster $j$ is the geometric mean of two fractions: the share of the box's frustum points that belong to $j$ and the share of $j$'s points that fall inside box $i$. The Hungarian algorithm (Kuhn, 1955) then assigns pairs one-to-one; pairs with a score below 0.2 are not associated.

Every object in the fused list needs a position. We compare five object lists:

| Object list | Existence from | Class from | Position from |
|---|---|---|---|
| **Camera** | camera | camera | flat-road monocular estimate |
| **LiDAR** | LiDAR cluster | — (class-agnostic) | cluster centroid |
| **Fusion AND** (confirmation) | camera *and* LiDAR | camera | LiDAR cluster |
| **Fusion camera-led** | camera | camera | LiDAR cluster if associated; else median depth of frustum points; else flat road |
| **Fusion OR** (union) | camera *or* LiDAR | camera if available | as camera-led, plus unassociated clusters |

In [ ]:
ASSOC_MIN, SHRINK = 0.2, 0.15

def to_rect_with(Tr, frame, pts):
    homo = np.c_[pts[:, :3], np.ones(len(pts))]
    return (frame.calib['R0'] @ Tr @ homo.T).T[:, :3]

def fuse_frame(frame, cam_dets, clusters, Tr=None):
    """Associate camera boxes and LiDAR clusters; return the five object lists and the matches."""
    Tr = frame.calib['Tr_velo_to_cam'] if Tr is None else Tr
    H, W = frame.image.shape[:2]
    cl = []
    for c in clusters:                                   # project clusters with the (possibly perturbed) extrinsic
        rect = to_rect_with(Tr, frame, c['points'])
        front = rect[:, 2] > 1.0
        if front.mean() < 0.5:
            continue
        uv = project_rect(frame, rect[front])
        inside = (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
        if inside.mean() < 0.5:
            continue
        cl.append({'uv': uv, 'rect': rect[front], 'ref': rect[front][:, [0, 2]].mean(0),
                   'box': np.r_[uv.min(0), uv.max(0)]})
    A = np.zeros((len(cam_dets), len(cl)))
    frustum_depth = []
    for i, d in enumerate(cam_dets):
        b = d['box']; w, h = b[2] - b[0], b[3] - b[1]
        ib = (b[0] + SHRINK * w, b[1] + SHRINK * h, b[2] - SHRINK * w, b[3] - SHRINK * h)
        inbox = [(c['uv'][:, 0] >= ib[0]) & (c['uv'][:, 0] <= ib[2]) & (c['uv'][:, 1] >= ib[1]) & (c['uv'][:, 1] <= ib[3])
                 for c in cl]
        counts = np.array([m.sum() for m in inbox])
        total = counts.sum()
        for j, c in enumerate(cl):
            if counts[j] >= 3:
                A[i, j] = np.sqrt(counts[j] / total * counts[j] / len(c['uv']))
        zs = np.concatenate([c['rect'][m, 2] for c, m in zip(cl, inbox)]) if cl else np.array([])
        frustum_depth.append(np.median(zs) if len(zs) >= 3 else None)
    matches = []
    if A.size:
        matches = [(i, j) for i, j in zip(*linear_sum_assignment(-A)) if A[i, j] >= ASSOC_MIN]
    mi, mj = dict(matches), {j for _, j in matches}

    def cam_point(i):
        if i in mi:
            return cl[mi[i]]['ref']
        if frustum_depth[i] is not None:
            b, P2 = cam_dets[i]['box'], frame.calib['P2']
            z = frustum_depth[i]
            return np.array([((b[0] + b[2]) / 2 - P2[0, 2]) * z / P2[0, 0], z])
        return flat_ground_point(frame, cam_dets[i]['box'])

    lists = {'Camera': [], 'LiDAR': [], 'Fusion AND': [], 'Fusion camera-led': [], 'Fusion OR': []}
    for i, d in enumerate(cam_dets):
        p = flat_ground_point(frame, d['box'])
        if p is not None:
            lists['Camera'].append({'ref': p, 'cls': d['cls'], 'box': d['box']})
        q = cam_point(i)
        if q is not None:
            lists['Fusion camera-led'].append({'ref': q, 'cls': d['cls'], 'box': d['box'], 'lidar_depth': i in mi})
        if i in mi:
            lists['Fusion AND'].append({'ref': cl[mi[i]]['ref'], 'cls': d['cls'], 'box': d['box']})
    for j, c in enumerate(cl):
        lists['LiDAR'].append({'ref': c['ref'], 'cls': None, 'box': c['box']})
    lists['Fusion OR'] = lists['Fusion camera-led'] + [{'ref': c['ref'], 'cls': None, 'box': c['box']}
                                                        for j, c in enumerate(cl) if j not in mj]
    return lists, matches, cl

lists, matches, cl_show = fuse_frame(frames[SHOW], cam[SHOW], lidar[SHOW])
fr = frames[SHOW]
fig, ax = plt.subplots(figsize=(15, 5.5))
ax.imshow(fr.image)
for j, c in enumerate(cl_show):
    ax.scatter(c['uv'][:, 0], c['uv'][:, 1], s=1, color=OKABE_ITO[2] if j in {jj for _, jj in matches} else 'lightgrey')
for i, d in enumerate(cam[SHOW]):
    b = d['box']; matched = i in dict(matches)
    ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=2,
                           edgecolor=CLASS_COLOURS[d['cls']], ls='-' if matched else ':'))
    if matched:
        z = cl_show[dict(matches)[i]]['ref'][1]
        ax.text(b[0], b[1] - 3, f"{d['cls']} {z:.1f} m", fontsize=8, color='white',
                bbox=dict(facecolor=CLASS_COLOURS[d['cls']], alpha=0.8, pad=1, lw=0))
ax.set_xlim(0, fr.image.shape[1]); ax.set_ylim(fr.image.shape[0], 0); ax.grid(False)
ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')
ax.set_title(f'Frame {fr.index}: camera boxes (solid = associated with a LiDAR cluster, label = LiDAR depth; dotted = not associated); '
             'green = associated cluster points, grey = other clusters', fontsize=9)
plt.tight_layout(); plt.show()
print(f"Frame {fr.index}: {len(cam[SHOW])} camera boxes, {len(cl_show)} clusters in the image, {len(matches)} associations")

### 5.2 Evaluation protocol

What a braking function needs is the position of the object's nearest face in the ground plane. We therefore evaluate in BEV:

- **Ground truth:** KITTI cars, vans, trucks (→ Vehicle), pedestrians and cyclists with KITTI difficulty Easy, Moderate or Hard, up to 50 m away. All other labels (difficulty "ignored", Misc, Tram, beyond 50 m) and the DontCare regions are *ignore regions*: detections there count neither as hits nor as false alarms.
- **Hit:** the reported position lies within **1.0 m of the labelled box footprint** (0 m if inside). One-to-one assignment with the Hungarian algorithm. A detection that is not assigned and not in an ignore region is a false positive — including duplicate detections of the same object.
- **Class:** for hits of object lists that report a class, we also check the class group.

In [ ]:
GATE_M, MAX_RANGE = 1.0, 50.0

def footprint_distance(p, lab):
    """Distance (m) from a BEV point (x, z, rectified camera frame) to a KITTI label footprint."""
    ry = lab['rotation_y']; dx, dz = p[0] - lab['x'], p[1] - lab['z']
    lx = np.cos(ry) * dx - np.sin(ry) * dz; lz = np.sin(ry) * dx + np.cos(ry) * dz
    return np.hypot(max(abs(lx) - lab['l'] / 2, 0), max(abs(lz) - lab['w'] / 2, 0))

def split_labels(frame):
    gts, ignore = [], []
    for l in frame.labels:
        d = np.hypot(l['x'], l['z'])
        if l['type'] in GROUP and D.kitti_difficulty(l) != 'Ignored' and d <= MAX_RANGE:
            gts.append({'label': l, 'group': GROUP[l['type']], 'distance_m': d, 'occluded': l['occluded']})
        else:
            ignore.append(l)
    return gts, ignore

def evaluate(dets, frame):
    gts, ignore = split_labels(frame)
    M = np.array([[footprint_distance(d['ref'], g['label']) for g in gts] for d in dets]).reshape(len(dets), len(gts))
    hits = []
    if M.size:
        hits = [(i, j) for i, j in zip(*linear_sum_assignment(M)) if M[i, j] <= GATE_M]
    hit_d = {i for i, _ in hits}
    fp = 0
    for i, d in enumerate(dets):
        if i in hit_d:
            continue
        ign = any(footprint_distance(d['ref'], l) <= GATE_M for l in ignore if l['type'] != 'DontCare')
        u, v = (d['box'][0] + d['box'][2]) / 2, (d['box'][1] + d['box'][3]) / 2
        ign |= any(l['x1'] <= u <= l['x2'] and l['y1'] <= v <= l['y2'] for l in ignore if l['type'] == 'DontCare')
        fp += not ign
    return hits, fp, gts

def run_all(Tr_fn=None, methods=None):
    """Fuse and evaluate all frames. Tr_fn(frame) may return a perturbed extrinsic."""
    gt_rows, summary, assoc = [], {}, {}
    for f, fr in frames.items():
        Tr = None if Tr_fn is None else Tr_fn(fr)
        lists, matches, _ = fuse_frame(fr, cam[f], lidar[f], Tr)
        assoc[f] = set(matches)
        for name, dets in lists.items():
            if methods is not None and name not in methods:
                continue
            hits, fp, gts = evaluate(dets, fr)
            s = summary.setdefault(name, {'TP': 0, 'FP': 0, 'GT': 0, 'class_ok': 0, 'class_reported': 0})
            s['TP'] += len(hits); s['FP'] += fp; s['GT'] += len(gts)
            for i, j in hits:
                if dets[i]['cls'] is not None:
                    s['class_reported'] += 1; s['class_ok'] += dets[i]['cls'] == gts[j]['group']
            hit_gt = {j: i for i, j in hits}
            for j, g in enumerate(gts):
                err = None
                if j in hit_gt:
                    p = dets[hit_gt[j]]['ref']
                    err = abs(np.hypot(*p) - g['distance_m'])
                gt_rows.append({'method': name, 'frame': f, 'group': g['group'], 'distance_m': g['distance_m'],
                                'occluded': g['occluded'], 'hit': j in hit_gt, 'range_error_m': err})
    res = pd.DataFrame(summary).T
    res['recall'] = res['TP'] / res['GT']
    res['precision'] = res['TP'] / (res['TP'] + res['FP'])
    res['class_accuracy'] = res['class_ok'] / res['class_reported'].replace(0, np.nan)
    return res, pd.DataFrame(gt_rows), assoc

t0 = time.time()
results, per_gt, assoc_ref = run_all()
print(f"Fusion and evaluation of {len(frames)} frames: {time.time() - t0:.1f} s")
display(results[['GT', 'TP', 'FP', 'recall', 'precision', 'class_accuracy']].astype(float).round(3))

In [ ]:
per_gt['range'] = pd.cut(per_gt['distance_m'], [0, 15, 30, 50], labels=['0-15 m', '15-30 m', '30-50 m'])
by_group = per_gt.pivot_table(index='method', columns='group', values='hit', aggfunc='mean').loc[results.index]
by_range = per_gt.pivot_table(index='range', columns='method', values='hit', aggfunc='mean', observed=True)[results.index]
n_group = per_gt[per_gt['method'] == 'Camera'].groupby('group').size()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name in results.index:
    axes[0].scatter(results.loc[name, 'recall'], results.loc[name, 'precision'], s=120, color=METHOD_COLOURS[name], label=name)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].set_title(f'Object lists on {len(frames)} KITTI frames (hit = within {GATE_M} m)'); axes[0].legend(fontsize=8)
xx = np.arange(len(by_group.columns)); w = 0.16
for k, name in enumerate(results.index):
    axes[1].bar(xx + (k - 2) * w, by_group.loc[name] * 100, w, color=METHOD_COLOURS[name], label=name)
axes[1].set_xticks(xx); axes[1].set_xticklabels([f"{g}\n(n={n_group[g]})" for g in by_group.columns])
axes[1].set_ylabel('Recall (%)'); axes[1].set_ylim(0, 105); axes[1].set_title('Recall by class group')
for name in results.index:
    axes[2].plot(by_range.index.astype(str), by_range[name] * 100, 'o-', color=METHOD_COLOURS[name], label=name)
axes[2].set_xlabel('Distance to object'); axes[2].set_ylabel('Recall (%)'); axes[2].set_ylim(0, 105)
axes[2].set_title('Recall by distance')
plt.tight_layout(); plt.show()

r = results
best_single = max(['Camera', 'LiDAR'], key=lambda m: r.loc[m, 'recall'])
print("--- Single sensor vs late fusion (text summary, measured) ---")
for name in r.index:
    ca = r.loc[name, 'class_accuracy']
    print(f"  {name:18s} recall {r.loc[name, 'recall']:.2f}  precision {r.loc[name, 'precision']:.2f}  "
          f"(TP {int(r.loc[name, 'TP'])}, FP {int(r.loc[name, 'FP'])}, GT {int(r.loc[name, 'GT'])})"
          + (f"  class accuracy of hits {ca:.2f}" if pd.notna(ca) else "  no class"))
print(f"  Highest single-sensor recall: {best_single} ({r.loc[best_single, 'recall']:.2f}). "
      f"Fusion OR reaches {r.loc['Fusion OR', 'recall']:.2f} at precision {r.loc['Fusion OR', 'precision']:.2f}; "
      f"Fusion AND reaches precision {r.loc['Fusion AND', 'precision']:.2f} at recall {r.loc['Fusion AND', 'recall']:.2f}.")
gain = r.loc['Fusion camera-led', 'recall'] - r.loc['Camera', 'recall']
print(f"  Giving camera boxes a LiDAR position (camera-led) changes BEV recall by {gain:+.2f} compared with the "
      "flat-road camera estimate: the camera finds the objects, but cannot place them.")
ped = by_group['Pedestrian']
print("  Pedestrian recall: " + ", ".join(f"{m} {ped[m]:.2f}" for m in r.index))

### 5.3 Where does the position come from?

The flat-road estimate breaks down where the road is not flat, where the bottom edge of the box is wrong (occlusion, truncation) and at range. Measured on all camera detections that hit an object, the range error of the camera-only position is compared with the position delivered by the LiDAR in the fused list.

In [ ]:
all_cam = []
for f, fr in frames.items():                 # every camera detection matched to a label in 2D (IoU >= 0.5)
    gts, _ = split_labels(fr)
    for d in cam[f]:
        p = flat_ground_point(fr, d['box'])
        ious = [iou2d(d['box'], [g['label'][k] for k in ('x1', 'y1', 'x2', 'y2')]) for g in gts]
        if p is not None and ious and max(ious) >= 0.5:
            all_cam.append((gts[int(np.argmax(ious))]['distance_m'], np.hypot(*p) - gts[int(np.argmax(ious))]['distance_m']))
all_cam = np.array(all_cam)
lid = per_gt[per_gt['hit'] & (per_gt['method'] == 'Fusion AND')]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(all_cam[:, 0], np.abs(all_cam[:, 1]), s=18, color=METHOD_COLOURS['Camera'], alpha=0.75,
           label='Camera, flat-road estimate (all 2D-matched detections)')
ax.scatter(lid['distance_m'], lid['range_error_m'], s=18, marker='s', color=METHOD_COLOURS['Fusion AND'], alpha=0.75,
           label='Fusion, LiDAR position (hits)')
ax.set_yscale('log'); ax.set_xlabel('Distance to object (m)'); ax.set_ylabel('|range error| to box centre (m, log)')
ax.set_title('Range error: camera flat-road estimate vs LiDAR position'); ax.legend()
plt.tight_layout(); plt.show()

print("--- Position source (text summary, measured) ---")
for lo, hi in [(0, 15), (15, 30), (30, 50)]:
    s = all_cam[(all_cam[:, 0] >= lo) & (all_cam[:, 0] < hi)]
    if len(s):
        print(f"  Camera, {lo}-{hi} m: n = {len(s)}, median |range error| {np.median(np.abs(s[:, 1])):.2f} m, "
              f"90th percentile {np.percentile(np.abs(s[:, 1]), 90):.2f} m")
print(f"  LiDAR position in fused hits: median |range error| to the box centre {lid['range_error_m'].median():.2f} m, "
      f"90th percentile {lid['range_error_m'].quantile(0.9):.2f} m (the LiDAR sees the near face; the centre lies "
      "half a box length behind it)")

---
## 6. Calibration error sensitivity

Fusion depends on the extrinsic calibration in two ways: the projection of LiDAR points into the image decides the **association**, and the transform into the common frame decides the **position** of the fused object. We perturb the real KITTI extrinsic by a small yaw (rotation about the LiDAR's vertical axis) or pitch error, keep the camera detections and LiDAR clusters fixed, and re-run association and evaluation. *Association retained* is the share of the original camera–cluster pairs that are still formed.

In [ ]:
def perturbed(axis, deg):
    R = np.eye(4)
    R[:3, :3] = Rotation.from_euler(axis, deg, degrees=True).as_matrix()   # rotation in the LiDAR frame
    return lambda fr: fr.calib['Tr_velo_to_cam'] @ R

sweep = []
n_ref = sum(len(v) for v in assoc_ref.values())
t0 = time.time()
for axis, name in [('z', 'yaw'), ('y', 'pitch')]:
    for deg in [0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
        res_e, _, assoc_e = run_all(perturbed(axis, deg), methods=('Fusion AND', 'Fusion camera-led'))
        kept = sum(len(assoc_ref[f] & assoc_e[f]) for f in frames) / max(n_ref, 1)
        sweep.append({'error': name, 'deg': deg, 'association retained': kept,
                      'AND recall': res_e.loc['Fusion AND', 'recall'], 'AND precision': res_e.loc['Fusion AND', 'precision'],
                      'camera-led recall': res_e.loc['Fusion camera-led', 'recall']})
sweep = pd.DataFrame(sweep)
base = {'association retained': 1.0, 'AND recall': results.loc['Fusion AND', 'recall'],
        'AND precision': results.loc['Fusion AND', 'precision'], 'camera-led recall': results.loc['Fusion camera-led', 'recall']}
print(f"Calibration sweep: {time.time() - t0:.0f} s")
display(sweep.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, metric in zip(axes, ['association retained', 'AND recall']):
    for name, col in [('yaw', OKABE_ITO[4]), ('pitch', OKABE_ITO[5])]:
        s = sweep[sweep['error'] == name]
        ax.plot(np.r_[0, s['deg']], np.r_[base[metric], s[metric]], 'o-', color=col, label=f'{name} error')
    ax.set_xlabel('Extrinsic rotation error (deg)'); ax.set_ylabel(metric); ax.set_ylim(0, 1.05); ax.legend()
axes[0].set_title('Share of camera-LiDAR associations still formed')
axes[1].set_title('Recall of the confirmed (AND) object list')
axes[1].axhline(results.loc['Camera', 'recall'], color=METHOD_COLOURS['Camera'], ls='--', label='Camera only')
axes[1].legend()
plt.tight_layout(); plt.show()

print("--- Calibration sensitivity (text summary, measured) ---")
for name in ['yaw', 'pitch']:
    s = sweep[sweep['error'] == name]
    half = s.loc[s['association retained'] < 0.5, 'deg']
    print(f"  {name}: association retained " + ", ".join(f"{d} deg -> {k:.2f}" for d, k in zip(s['deg'], s['association retained']))
          + f"; below 50 % from {half.min() if len(half) else '> 5'} deg")
fx = frames[SHOW].calib['P2'][0, 0]
print(f"  1 deg at fx = {fx:.0f} px shifts a projected point by ~{fx * np.tan(np.radians(1)):.0f} px - "
      "a pedestrian 30 m away is about 20 px wide.")

### Time synchronisation

If camera and LiDAR sample the scene at different instants, a moving object appears at different positions: $\Delta x = v_{rel}\,\Delta t$. KITTI triggers the cameras when the rotating LiDAR faces forward (Geiger et al., 2013), so the offset is near zero at the image centre but not at the image edges: the LiDAR needs time to sweep across the camera's field of view. Production systems use hardware time synchronisation (PTP / gPTP, IEEE 802.1AS), trigger cameras per sector (as in nuScenes) and **motion-compensate** the points with odometry.

In [ ]:
P2 = frames[SHOW].calib['P2']
half_fov = np.degrees(np.arctan(max(P2[0, 2], frames[SHOW].image.shape[1] - P2[0, 2]) / P2[0, 0]))
dt_edge = half_fov / 360 / 10.0                         # 10 Hz rotation
print(f"KITTI camera half field of view {half_fov:.1f} deg -> the LiDAR reaches the image edge "
      f"{dt_edge * 1000:.1f} ms before/after the camera trigger.")
dts = np.array([1, dt_edge * 1000, 33, 50, 100]) / 1000.0
v_rel = np.array([5, 14, 30])            # m/s: cyclist, urban traffic, oncoming motorway traffic (~108 km/h)
tbl = pd.DataFrame({f'{v} m/s': np.round(v * dts, 3) for v in v_rel}, index=[f'{dt*1000:.1f} ms' for dt in dts])
display(tbl)
print("Position error (m) = relative speed x time offset:")
for dt in dts:
    print(f"  {dt*1000:5.1f} ms: " + ", ".join(f"{v} m/s -> {v*dt:5.2f} m" for v in v_rel))
print(f"At the image edge in KITTI a crossing cyclist at 5 m/s is displaced by {5 * dt_edge:.2f} m between LiDAR and camera; "
      f"a 100 ms offset at 30 m/s is a {30*0.1:.1f} m error - larger than a car's width.")

### Other challenges
- **Conflicting information:** class conflicts (car vs truck) and existence conflicts (camera sees, LiDAR does not) need explicit, verified rules.
- **Sensor degradation:** dirty lenses, water, mud, ice — the system must detect degradation and down-weight or exclude the sensor.
- **Computational cost:** fusion must fit the latency budget (order of 100 ms end to end).
- **Common causes:** fusion of two sensors that fail together (e.g. both blinded by spray) gives no redundancy.

---
## Exercise: design a fusion strategy

**Scenario:** an L4 campus shuttle with 4 cameras (front, rear, left, right), 1 roof LiDAR (360°) and 2 radars (front, rear).

1. Which fusion approach would you use (early / late / intermediate) and why?
2. How do you detect and handle sensor failures and degradation?
3. How do you resolve class and existence conflicts — in particular for pedestrians?
4. How do you validate the fusion, including calibration and time-sync tolerances?

In [ ]:
your_strategy = {
    'fusion_approach': '',      # early / late / intermediate
    'reasoning': '',
    'failure_handling': '',
    'conflict_resolution': '',
    'validation_plan': '',
}
print("Consider: real-time constraints; safety criticality (ASIL from your HARA); sensor failure modes;"
      " environmental conditions; independence of sensor paths.")
# Sample answer (uncomment to see)
# sample_answer = {
#     'fusion_approach': 'BEV feature fusion for performance plus an independent late-fusion/LiDAR-cluster safety path',
#     'reasoning': 'Best accuracy from BEV fusion; an independent, simpler path supports the safety argument (diversity).',
#     'failure_handling': 'Sensor health monitoring, calibration monitoring, degraded modes, minimal-risk manoeuvre',
#     'conflict_resolution': 'Class-agnostic association; VRU-preferring rules; low single-sensor thresholds for VRUs + tracking confirmation',
#     'validation_plan': 'Per-class/per-distance metrics on nuScenes-like data + campus data, calibration/time-offset sweeps, scenario simulation, field tests',
# }
# print(sample_answer)

---
## Safety perspective

- **ISO 26262-9:2018** allows **ASIL decomposition** of a safety requirement onto redundant elements (e.g. D → B(D) + B(D)) only if the elements are **sufficiently independent** (dependent-failure analysis). Shared calibration, shared time base, shared compute or a shared learned BEV backbone are dependencies that must be analysed. Section 6 shows that a single shared parameter, the extrinsic calibration, can remove most associations at once.
- **ISO 21448:2022:** fusion is a measure against **performance insufficiencies** (one sensor's limitation is covered by another). Section 5 showed that the choice of fusion logic decides which insufficiency remains: confirmation (AND) suppresses false alarms but inherits the misses of both sensors; union (OR) inherits the false alarms of both. For the course's AEB example, the AND list is the natural candidate for SG-01 (avoid unintended braking, ASIL D) and the OR list for SG-02 (avoid omitted braking, ASIL B) — one fused list cannot serve both goals with the same thresholds.
- **ISO/PAS 8800:2024:** the camera detector here is an AI component trained on COCO, not on the target ODD; its data coverage (KITTI-like streets, cyclists, partly occluded pedestrians) must be argued or measured — exactly what section 5 does in miniature.
- Association thresholds, frustum shrink factors and gating distances are **safety-relevant design parameters**: derive them from requirements and verify their sensitivity.
- Calibration files and time bases are also **security assets**: a manipulated extrinsic has the same effect as the errors in section 6. The author's GSN pattern integrates such cybersecurity claims into the assurance argument of AI-based perception components (Patel and Jung, 2026).

## In industry

**Work product:** inputs to the **design FMEA of the fusion component** and to its **calibration validation plan**; produced by the **perception/fusion architect** with the **functional safety engineer**, reviewed in the DFMEA team and by the safety assessor. The measurements in this notebook map directly onto FMEA rows:

| Function | Failure mode | Effect on vehicle level | Cause (measured here) | Detection / prevention control |
|---|---|---|---|---|
| Associate camera box and LiDAR cluster | No association | Object unconfirmed → AEB does not brake (SG-02) | Extrinsic yaw/pitch error (section 6); object merged with background cluster | Online calibration monitoring; tracking-based confirmation |
| Associate | Wrong association | Wrong distance for a real object | Frustum contains background or neighbouring objects | Shrunk frustum, depth-consistency check |
| Estimate position | Range error | Late or premature braking | Flat-road assumption for camera-only objects (section 5.3) | Use LiDAR depth; plausibility of camera depth |
| Report existence | False object | Unintended braking (SG-01) | Unassociated clusters (poles, walls) or parked bicycles in the OR list | Confirmation logic for braking decisions |
| Time alignment | Offset between sensors | Position error v·Δt | Sweep timing at image edges (section 6) | Hardware sync (PTP), motion compensation |

What reviewers look for:
- Fusion metrics stratified by class, distance and occlusion, computed against ground truth with stated ignore rules and matching gates.
- A calibration validation procedure with measured tolerance (how many degrees before association degrades) and an online monitor with a defined reaction.
- An explicit argument for which fused list (AND / OR / tracked) feeds which safety goal.
- Evidence of independence or a dependent-failure analysis for any decomposition across the camera and LiDAR paths.

Continue with: [07 Multi-object tracking](07_multi_object_tracking.ipynb) · [03_Functional_Safety_ISO26262/03 — FMEA and FMEDA](../03_Functional_Safety_ISO26262/03_fmea_and_fmeda.ipynb) · [04_SOTIF_ISO21448/01 — SOTIF fundamentals](../04_SOTIF_ISO21448/01_sotif_fundamentals.ipynb).

## Key takeaways

- On real KITTI frames, neither single sensor is sufficient: the COCO-pretrained camera detector finds and classifies objects but cannot place them reliably with a flat-road estimate; LiDAR clusters place objects precisely but carry no class and include many unlabelled structures.
- Late fusion is a family of decisions, not one result: confirmation (AND) trades recall for precision, union (OR) trades precision for recall, and camera-led fusion with LiDAR depth combines the camera's semantics with the LiDAR's geometry. The printed numbers in section 5 quantify each trade-off.
- Frustum association with real calibration works out of the box on KITTI; projected 3D labels agree closely with the annotated 2D boxes.
- A rotation error of the extrinsic calibration of about one degree already removes a large share of associations (section 6): calibration validation and online monitoring are safety mechanisms, not housekeeping.
- Even a well-triggered rig has a time offset at the image edges (about 11 ms in KITTI); offsets translate into position errors v·Δt.
- The measured failure modes are direct inputs to the fusion DFMEA and to the SOTIF analysis.

## Self-assessment

1. What makes late fusion attractive for modular, supplier-based production systems?
2. In section 5, why does the AND list have higher precision but lower recall than either the camera or the LiDAR list?
3. Why does the camera-only list score poorly in a BEV evaluation even when the 2D boxes are correct?
4. From the calibration sweep: from roughly which yaw error is less than half of the associations retained? Why is pitch error also critical for a camera whose image is only 375 px high?
5. Name two failure modes that sensor fusion cannot solve.
6. Which fused list would you feed to an AEB function's braking decision, and how do the two AEB safety goals (unintended vs omitted braking) pull in opposite directions?

<details>
<summary>Show answers</summary>

1. Each sensor supplier delivers a validated object list; interfaces are simple; paths can be tested and replaced independently, and the rules are explainable.
2. AND keeps an object only if both sensors report it: false alarms of one sensor are suppressed unless the other also fires, but every miss of either sensor (and every failed association) becomes a miss of the fused list.
3. The flat-road estimate converts small errors in the box's bottom edge, occlusion of the lower body and road slope into large range errors, which grow with distance; a correct 2D box can still be metres away from the true position.
4. Read it from the printed sweep (in this run about 1–2°). Vertical misalignment moves projected points up or down; the boxes of distant pedestrians are only some tens of pixels high, so a fraction of a degree moves the LiDAR points out of the box.
5. Common-cause degradation of all sensors (heavy snow, spray, shared power or compute failure, a shared calibration error), and objects outside every sensor's capability (unknown object types the camera was not trained on and that are too small or too occluded for the LiDAR).
6. A braking decision needs high confidence that an object exists (against SG-01, unintended braking, ASIL D), which favours confirmed (AND) objects, typically confirmed further by tracking over several frames. Avoiding omitted braking (SG-02) favours keeping single-sensor objects. Practical systems combine both: tracked objects from the union, with braking authority graded by how well each object is confirmed.

</details>

## References

1. ISO 26262-9:2018. *Road vehicles — Functional safety — Part 9: ASIL-oriented and safety-oriented analyses.* https://www.iso.org/standard/68391.html
2. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
3. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence.*
4. Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. *CVPR 2012*. https://doi.org/10.1109/CVPR.2012.6248074
5. Geiger, A., Lenz, P., Stiller, C., Urtasun, R. (2013). Vision meets robotics: The KITTI dataset. *The International Journal of Robotics Research* 32(11), 1231–1237. https://doi.org/10.1177/0278364913491297
6. Ren, S., He, K., Girshick, R., Sun, J. (2017). Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks. *IEEE TPAMI* 39(6), 1137–1149. https://doi.org/10.1109/TPAMI.2016.2577031
7. Howard, A. et al. (2019). Searching for MobileNetV3. *ICCV 2019*. https://doi.org/10.1109/ICCV.2019.00140
8. Lin, T.-Y. et al. (2014). Microsoft COCO: Common Objects in Context. *ECCV 2014*, LNCS 8693. https://doi.org/10.1007/978-3-319-10602-1_48
9. Qi, C. R., Liu, W., Wu, C., Su, H., Guibas, L. J. (2018). Frustum PointNets for 3D Object Detection from RGB-D Data. *CVPR 2018*. https://doi.org/10.1109/CVPR.2018.00102
10. Vora, S., Lang, A. H., Helou, B., Beijbom, O. (2020). PointPainting: Sequential Fusion for 3D Object Detection. *CVPR 2020*. https://doi.org/10.1109/CVPR42600.2020.00466
11. Li, Z. et al. (2022). BEVFormer: Learning Bird's-Eye-View Representation from Multi-Camera Images via Spatiotemporal Transformers. ECCV. arXiv:2203.17270
12. Liu, Z. et al. (2023). BEVFusion: Multi-Task Multi-Sensor Fusion with Unified Bird's-Eye View Representation. ICRA. arXiv:2205.13542
13. Bai, X. et al. (2022). TransFusion: Robust LiDAR-Camera Fusion for 3D Object Detection with Transformers. CVPR. arXiv:2203.11496
14. Zhang, Z. (2000). A flexible new technique for camera calibration. *IEEE TPAMI* 22(11), 1330–1334. https://doi.org/10.1109/34.888718
15. Caesar, H. et al. (2020). nuScenes: A multimodal dataset for autonomous driving. *CVPR 2020*. https://doi.org/10.1109/CVPR42600.2020.01164
16. Kuhn, H. W. (1955). The Hungarian method for the assignment problem. *Naval Research Logistics Quarterly* 2, 83–97. https://doi.org/10.1002/nav.3800020109
17. Patel, M., Jung, R. (2026). Integrating Cybersecurity Into the AI Safety Assurance Argument: A GSN Pattern for AI-Based Perception Components in Highly Automated Driving. In *Computer Safety, Reliability, and Security. SAFECOMP 2026 Workshops*, LNCS, pp. 763–779. Springer. https://doi.org/10.1007/978-3-032-35506-5_55

### Further practice

Companion hands-on material: [AI Perception Safety Workshop — Notebook 01, Sensor overview](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/01_sensor_overview.ipynb).

---
**Navigation:** ← [05 Vulnerable road user detection](05_vulnerable_road_user_detection.ipynb) · [Module index](README.md) · [07 Multi-object tracking](07_multi_object_tracking.ipynb) →